# Spotify Music Genre Classification

This notebook tests whether Spotify audio characteristics can predict a song's genre. A Random Forest classifier is evaluated across five genres using stratified five-fold cross-validation.


## 1. Imports

In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_validate
)


## 2. Load the dataset

The dataset contains Spotify tracks and their audio characteristics. The first five records are displayed to confirm that the file has loaded correctly and to inspect its structure.


In [3]:
data = pd.read_csv("spotifyMusicGenres.csv")
data.head()

,artist,song,ids,genre,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,speechiness,tempo,mode,valence
0,Kate Bush,Running Up That Hill (A Deal With God),75FEaRjZTKLhTrFGsfMUXR,Rock,0.72000,0.629,298933,0.547,0.003140,10,0.0604,-13.123,0.0550,108.375,0,0.197
1,The Killers,Mr. Brightside,003vvx7Niy0yvhvHt4a68B,Rock,0.00121,0.352,222973,0.911,0.000000,1,0.0995,-5.230,0.0747,148.033,1,0.236
2,Arctic Monkeys,505,0BxE4FqsDD1Ot4YuBXwAPp,Rock,0.00287,0.526,253587,0.866,0.000078,0,0.0945,-5.822,0.0568,140.266,1,0.248
3,Sam Fender,Seventeen Going Under,5rF6YUIlgiat22OT1lWspJ,Rock,0.00438,0.480,297933,0.870,0.006030,1,0.0826,-4.792,0.0362,161.953,1,0.584
4,George Ezra,Green Green Grass,3rk4aJ0vAj3cFUIQEeASkT,Rock,0.06950,0.685,167614,0.738,0.000000,8,0.1280,-4.413,0.0595,112.972,1,0.800


## 3. Data-quality checks

Before modelling, the dataset is checked for missing values and the available target genres are reviewed. No missing values are present, so no imputation or row removal is required.


In [4]:
data.isnull().sum()

artist              0
song                0
ids                 0
genre               0
acousticness        0
danceability        0
duration_ms         0
energy              0
instrumentalness    0
key                 0
liveness            0
loudness            0
speechiness         0
tempo               0
mode                0
valence             0
dtype: int64

In [5]:
data['genre'].unique()

<ArrowStringArray>
['Rock', 'Rap', 'Country', 'Classical', 'EDM']
Length: 5, dtype: str

## 4. Select the model features and target

The model uses numerical Spotify audio characteristics as inputs. Artist name, song title, and Spotify ID are excluded because they identify individual tracks rather than describing their sound. The target variable is `genre`.


In [ ]:
# Features and target
feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "key",
    "liveness",
    "loudness",
    "mode",
    "speechiness",
    "tempo",
    "valence"
]

X = data[feature_columns]
y = data["genre"]

## 5. Configure the Random Forest and cross-validation

Random Forest is suitable for this task because it supports multiclass classification and can capture non-linear relationships between audio features.

Five-fold stratified cross-validation is used because the dataset is small. Each fold maintains the genre proportions, allowing every record to be used for both training and validation while preventing it from appearing in both during the same fold.

The model is evaluated using:

- **Accuracy:** the overall proportion of correctly classified songs.
- **Macro F1:** the average F1-score across genres, giving every genre equal importance.


In [ ]:
# Random Forest model
model = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Five-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Calculate cross-validation metrics
scores = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro"
    },
    n_jobs=-1
)

## 6. Generate out-of-fold predictions

`cross_val_predict` produces one prediction for every song from a model that was not trained on that song. These predictions are used to generate the classification report.


In [ ]:
# Generate predictions for every record
y_pred = cross_val_predict(
    model,
    X,
    y,
    cv=cv,
    n_jobs=-1
)

## 7. Evaluate the model

The cross-validation averages provide an overall performance estimate. The classification report then shows precision, recall, and F1-score for each individual genre.


In [21]:
print(f"Mean accuracy: {scores['test_accuracy'].mean():.3f}")
print(f"Mean macro F1: {scores['test_macro_f1'].mean():.3f}")

print("\nClassification report:")
print(classification_report(y, y_pred, digits=3))


Mean accuracy: 0.724
Mean macro F1: 0.712

Classification report:
              precision    recall  f1-score   support

   Classical      0.980     0.960     0.970        50
     Country      0.643     0.720     0.679        50
         EDM      0.647     0.660     0.653        50
         Rap      0.754     0.860     0.804        50
        Rock      0.568     0.420     0.483        50

    accuracy                          0.724       250
   macro avg      0.718     0.724     0.718       250
weighted avg      0.718     0.724     0.718       250



## 8. Interpretation and dataset decision

The Random Forest achieved a mean accuracy of **72.4%** and a mean macro F1-score of **71.2%**, which is promising for a dataset containing only 250 songs.

Performance varies substantially by genre. Classical music is identified particularly well, whereas Rock has the lowest F1-score and is more difficult to distinguish using these audio characteristics. This suggests that some genre boundaries overlap.

However, the dataset contains only five genres with 50 songs per genre. Although this makes it useful for an initial experiment, it does not represent the wider range of music the application may encounter. A larger and more diverse dataset is therefore more appropriate for training the final genre model.
